In [1]:
import pandas as pd
from collections import defaultdict
from pyswmm import Simulation, Links

inp_path = r"C:\Project\Avon\Hydrologic\NewWork_WDT\Avon_setup_final.inp"

# ----- Build upstream connectivity -----
node_to_links = defaultdict(list)
conduit_inlet = {}

with Simulation(inp_path) as sim:
    for link in Links(sim):
        if link.is_conduit:
            conduit_inlet[link.linkid] = link.inlet_node
            node_to_links[link.outlet_node].append(link.linkid)  # Links ending at node

# upstream lookup table: link -> list of upstream links
conduit_upstream = defaultdict(list)
for conduit, node in conduit_inlet.items():
    conduit_upstream[conduit] = node_to_links.get(node, [])

# ----- Strahler -----
link_order = {}

def compute_order(conduit):
    if conduit in link_order:
        return link_order[conduit]

    ups = conduit_upstream.get(conduit, [])

    if not ups:
        order = 1
    else:
        orders = [compute_order(u) for u in ups]
        m = max(orders)
        order = m + 1 if orders.count(m) > 1 else m

    link_order[conduit] = order
    return order

# Compute all
for c in conduit_inlet:
    compute_order(c)

# ----- Export -----
df = pd.DataFrame(list(link_order.items()), columns=["LinkID", "StrahlerOrder"])
output = r"C:\Project\Avon\Hydrologic\NewWork_WDT\strahler_orders.xlsx"
df.to_excel(output, index=False)
print(f"Done → {output}")


Done → C:\Project\Avon\Hydrologic\NewWork_WDT\strahler_orders.xlsx
